<a href="https://colab.research.google.com/github/shreyansh8h/hello-world/blob/master/Copy_of_Lang_Chain_Introduction_April2026_Jothi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-core langchain-community langchain-huggingface transformers sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


Import the libraries

In [ ]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
generation_pipeline = pipeline(
    task="text-generation",
    model=MODEL_ID,
    tokenizer=MODEL_ID,
    max_new_tokens=256,
    do_sample=False,
)
hf_llm = HuggingFacePipeline(pipeline=generation_pipeline)
chat_model = ChatHuggingFace(llm=hf_llm)
print("Loaded:", MODEL_ID)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.02 GiB. GPU 0 has a total capacity of 14.56 GiB of which 59.81 MiB is free. Including non-PyTorch memory, this process has 14.50 GiB memory in use. Of the allocated memory 14.14 GiB is allocated by PyTorch, and 238.51 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

LangChain Libraries Import

In [ ]:
from pathlib import Path
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import FileChatMessageHistory
from langchain_core.output_parsers import StrOutputParser

In [ ]:
memory_dir = Path("chat_memory")
memory_dir.mkdir(exist_ok=True)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert in football and have a very good understanding on FIFA worldcups, If you dont know the answer say I dont know"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ]
)

chat_chain = prompt | chat_model | StrOutputParser()

def get_history(session_id: str) -> FileChatMessageHistory:
    return FileChatMessageHistory(str(memory_dir / f"{session_id}.json"))

chatbot = RunnableWithMessageHistory(
    chat_chain,
    get_session_history=get_history,
    input_messages_key="input",
    history_messages_key="history",
)

def ask(text: str, session_id: str = "demo-user") -> str:
    return chatbot.invoke(
        {"input": text},
        config={"configurable": {"session_id": session_id}},
    )

Initialize a chat

In [ ]:
session_id = "demo-user"

print("Q1:", ask("Who won the 2018 FIFA World Cup?", session_id=session_id))
print("Q2:", ask("What was the final score?", session_id=session_id))

memory_file = memory_dir / f"{session_id}.json"
print("Memory saved at:", memory_file.resolve())
print("Memory file exists:", memory_file.exists())

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q1: <|im_start|>system
You are an expert in football and have a very good understanding on FIFA worldcups, If you dont know the answer say I dont know<|im_end|>
<|im_start|>user
Who won the 2018 FIFA World Cup?<|im_end|>
<|im_start|>assistant
The 2018 FIFA World Cup was won by Argentina. They defeated Germany in the final with a score of 3-1.
Q2: <|im_start|>system
You are an expert in football and have a very good understanding on FIFA worldcups, If you dont know the answer say I dont know<|im_end|>
<|im_start|>user
Who won the 2018 FIFA World Cup?<|im_end|>
<|im_start|>assistant
<|im_start|>system
You are an expert in football and have a very good understanding on FIFA worldcups, If you dont know the answer say I dont know<|im_end|>
<|im_start|>user
Who won the 2018 FIFA World Cup?<|im_end|>
<|im_start|>assistant
The 2018 FIFA World Cup was won by Argentina. They defeated Germany in the final with a score of 3-1.<|im_end|>
<|im_start|>user
What was the final score?<|im_end|>
<|im_sta

In [ ]:
print("Q3:", ask("Who did they beat in the final?", session_id=session_id))

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q3: <|im_start|>system
You are an expert in football and have a very good understanding on FIFA worldcups, If you dont know the answer say I dont know<|im_end|>
<|im_start|>user
Who won the 2018 FIFA World Cup?<|im_end|>
<|im_start|>assistant
<|im_start|>system
You are an expert in football and have a very good understanding on FIFA worldcups, If you dont know the answer say I dont know<|im_end|>
<|im_start|>user
Who won the 2018 FIFA World Cup?<|im_end|>
<|im_start|>assistant
The 2018 FIFA World Cup was won by Argentina. They defeated Germany in the final with a score of 3-1.<|im_end|>
<|im_start|>user
What was the final score?<|im_end|>
<|im_start|>assistant
<|im_start|>system
You are an expert in football and have a very good understanding on FIFA worldcups, If you dont know the answer say I dont know<|im_end|>
<|im_start|>user
Who won the 2018 FIFA World Cup?<|im_end|>
<|im_start|>assistant
<|im_start|>system
You are an expert in football and have a very good understanding on FIFA 

### Tooling

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
    "Multiply two integers."
    return a * b

@tool
def world_cup_winner(year: int) -> str:
    "Return FIFA World Cup winner for a few known years."
    winners = {
        2014: "Germany",
        2018: "France",
        2022: "Argentina",
    }
    return winners.get(year, "I only know 2014, 2018, and 2022 in this demo.")

### Creating an Agent

In [ ]:
agent = create_agent(
    model=chat_model,
    tools=[multiply, world_cup_winner],
    system_prompt="You are a helpful assistant.Use world_cup_winner tool when question is about FIFA world cup winner. Use the multiply tool when asked to multiply",
)

In [ ]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is 2350 * 4950? Also, who was the winner of the 2018 FIFA World Cup?",
            }
        ]
    }
)

print(response["messages"][-1].content)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>system
You are a helpful assistant.Use world_cup_winner tool when question is about FIFA world cup winner. Use the multiply tool when asked to multiply<|im_end|>
<|im_start|>user
What is 2350 * 4950? Also, who was the winner of the 2018 FIFA World Cup?<|im_end|>
<|im_start|>assistant
The product of 2350 and 4950 is 11672500.
The winner of the 2018 FIFA World Cup was France.
